<a href="https://colab.research.google.com/github/hpatel1933/AAI2025/blob/main/Exercise_2_ReACT_Code_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 2: ReACT-Style Python Code Generation

Goal: generate a student average and letter-grade program using Plan, Generate, Run, Observe, Fix, and Final stages. Tools: Google Colab and Python. The code below shows the prompt, working implementation, normal tests, and edge-case tests.

In [2]:
react_prompt = "Plan, generate, run, observe, fix, and present final working Python code for a student average and letter grade. Handle empty lists and invalid scores."
print("TASK PROMPT:", react_prompt)

plan = ["Plan: validate input, calculate average, map average to A-F.", "Generate: produce a first implementation.", "Run: execute a normal sample.", "Observe: test empty and out-of-range inputs.", "Fix: add validation and clear error results.", "Final: execute the revised implementation and report evidence."]
for stage in plan:
    print(stage)

# Simulate the generated draft as executable Python source.
initial_code = '''def grade_student(scores):
    average = sum(scores) / len(scores)
    return average
'''
initial_namespace = {}
exec(initial_code, initial_namespace)
print("\nGENERATED DRAFT OUTPUT:", initial_namespace["grade_student"]([90, 85, 95]))
try:
    initial_namespace["grade_student"]([])
except ZeroDivisionError as error:
    print("OBSERVE: Edge-case failure detected:", type(error).__name__)

# The ReACT fix is generated from the observed failure.
final_code = '''def grade_student(scores):
    if not scores:
        return None, "No grade: the score list is empty."
    if any(not isinstance(score, (int, float)) or score < 0 or score > 100 for score in scores):
        return None, "Invalid input: every score must be between 0 and 100."
    average = sum(scores) / len(scores)
    if average >= 90: letter = "A"
    elif average >= 80: letter = "B"
    elif average >= 70: letter = "C"
    elif average >= 60: letter = "D"
    else: letter = "F"
    return average, letter
'''
final_namespace = {}
exec(final_code, final_namespace)

samples = [[90, 85, 95], [], [90, 105, 80]]
results = []
for sample in samples:
    results.append((sample, final_namespace["grade_student"](sample)))
print("\nOBSERVE AND FIX: Added empty-list and 0-100 validation after the draft failed on an empty list.")
print("FINAL WORKING CODE:\n", final_code)
print("FINAL TEST RESULTS:")
for sample, result in results:
    print(sample, "=>", result)
assert results[0][1] == (90.0, "A")
assert results[1][1][1].startswith("No grade")
assert results[2][1][1].startswith("Invalid input")
print("REACT VALIDATION: PASS - plan, generation, execution, observation, fix, and final tests completed.")

TASK PROMPT: Plan, generate, run, observe, fix, and present final working Python code for a student average and letter grade. Handle empty lists and invalid scores.
Plan: validate input, calculate average, map average to A-F.
Generate: produce a first implementation.
Run: execute a normal sample.
Observe: test empty and out-of-range inputs.
Fix: add validation and clear error results.
Final: execute the revised implementation and report evidence.

GENERATED DRAFT OUTPUT: 90.0
OBSERVE: Edge-case failure detected: ZeroDivisionError

OBSERVE AND FIX: Added empty-list and 0-100 validation after the draft failed on an empty list.
FINAL WORKING CODE:
 def grade_student(scores):
    if not scores:
        return None, "No grade: the score list is empty."
    if any(not isinstance(score, (int, float)) or score < 0 or score > 100 for score in scores):
        return None, "Invalid input: every score must be between 0 and 100."
    average = sum(scores) / len(scores)
    if average >= 90: letter